# 04 — Export clean data for Tableau Desktop

Phase 5 deliverable: two flat CSVs that Tableau can connect to directly.

| File                          | Grain                         | Used for |
|-------------------------------|-------------------------------|----------|
| `tableau/cohort_matrix.csv`   | one row per (cohort, month)   | retention heatmap |
| `tableau/customer_summary.csv`| one row per customer          | KPI cards, country bars, segment filters |

Both files are written into a dedicated `tableau/` folder so the Tableau workbook has its own clean data folder to point at.

## 0. Setup

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_DIR    = PROJECT_ROOT / "data"   / "clean"
SQL_DIR      = PROJECT_ROOT / "sql"
TABLEAU_DIR  = PROJECT_ROOT / "tableau"
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE VIEW transactions AS
    SELECT * FROM read_parquet('{(CLEAN_DIR / "online_retail_clean.parquet").as_posix()}');
""")
con.execute((SQL_DIR / "00_customer_summary.sql").read_text(encoding="utf-8"))
print("customers view ready:", con.execute("SELECT COUNT(*) FROM customers").fetchone()[0], "rows")

customers view ready: 5878 rows


## 1. `cohort_matrix.csv` — retention % long-form

**Long format on purpose.** Tableau can pivot a wide matrix into long, but it's annoying — every new column makes the pivot heavier. A long table (one row per `cohort_month × cohort_index`) is the format every Tableau cohort tutorial assumes.

Columns:

| column            | type    | meaning |
|-------------------|---------|---------|
| `cohort_month`    | DATE    | month each customer in this cohort first purchased |
| `cohort_index`    | INT     | months since cohort_month (0 = first month) |
| `cohort_size`     | INT     | unique customers in the cohort (constant per cohort) |
| `active_customers`| INT     | unique customers from the cohort active in this month |
| `retention_pct`   | FLOAT   | active_customers / cohort_size × 100 |

In [2]:
cohort_matrix = con.execute("""
    WITH labelled AS (
        SELECT
            t.CustomerID,
            DATE_TRUNC('month', c.first_purchase_date)::DATE                                          AS cohort_month,
            CAST(DATE_DIFF('month',
                           DATE_TRUNC('month', c.first_purchase_date),
                           DATE_TRUNC('month', t.InvoiceDate)) AS INT)                                AS cohort_index
        FROM transactions t
        JOIN customers   c ON t.CustomerID = c.CustomerID
    ),
    counts AS (
        SELECT cohort_month, cohort_index, COUNT(DISTINCT CustomerID) AS active_customers
        FROM labelled
        GROUP BY cohort_month, cohort_index
    ),
    sizes AS (
        SELECT cohort_month, active_customers AS cohort_size
        FROM counts
        WHERE cohort_index = 0
    )
    SELECT
        c.cohort_month,
        c.cohort_index,
        s.cohort_size,
        c.active_customers,
        ROUND(100.0 * c.active_customers / s.cohort_size, 2) AS retention_pct
    FROM counts c
    JOIN sizes  s USING (cohort_month)
    ORDER BY c.cohort_month, c.cohort_index;
""").df()

print("shape:", cohort_matrix.shape)
cohort_matrix.head(10)

shape:

 (325, 5)


,cohort_month,cohort_index,cohort_size,active_customers,retention_pct
0,2009-12-01,0,955,955,100.00
1,2009-12-01,1,955,337,35.29
2,2009-12-01,2,955,319,33.40
3,2009-12-01,3,955,406,42.51
4,2009-12-01,4,955,363,38.01
5,2009-12-01,5,955,343,35.92
6,2009-12-01,6,955,360,37.70
7,2009-12-01,7,955,327,34.24
8,2009-12-01,8,955,321,33.61
9,2009-12-01,9,955,346,36.23


In [3]:
out = TABLEAU_DIR / "cohort_matrix.csv"
cohort_matrix.to_csv(out, index=False, date_format="%Y-%m-%d")
print("Saved →", out, f"({out.stat().st_size/1024:.1f} KB)")

Saved → C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis\tableau\cohort_matrix.csv (8.7 KB)


## 2. `customer_summary.csv` — one row per customer

Columns the brief asked for: cohort month, total orders, total revenue, churned yes/no, country, avg order value. Plus a few extras that fall out for free and will be handy in Tableau filters.

**Churn flag — definition.** A customer is `is_churned = 1` if they placed exactly one order in their lifetime (the brief's definition: "bought once, never came back"). We also include `is_lapsed_90d` (no purchase in the last 90 days of the dataset) and `retained_90d` (from Phase 4) so you can swap definitions in Tableau without re-exporting.

In [4]:
customer_summary = con.execute("""
    WITH bounds AS (SELECT MAX(InvoiceDate) AS max_date FROM transactions)
    SELECT
        c.CustomerID                                                            AS customer_id,
        c.first_purchase_month                                                  AS cohort_month,
        c.first_country                                                         AS country,
        c.first_purchase_date::DATE                                             AS first_purchase_date,
        c.last_purchase_date::DATE                                              AS last_purchase_date,
        DATE_DIFF('day', c.first_purchase_date, c.last_purchase_date)           AS customer_lifespan_days,
        c.total_orders,
        ROUND(c.lifetime_revenue, 2)                                            AS total_revenue,
        ROUND(c.lifetime_revenue / NULLIF(c.total_orders, 0), 2)                AS avg_order_value,
        ROUND(c.first_order_value, 2)                                           AS first_order_value,
        CASE WHEN c.total_orders = 1 THEN 1 ELSE 0 END                          AS is_churned,
        CASE WHEN c.last_purchase_date < b.max_date - INTERVAL 90 DAY
             THEN 1 ELSE 0 END                                                  AS is_lapsed_90d,
        CASE WHEN c.eligible_for_90d THEN 1 ELSE 0 END                          AS eligible_for_90d,
        CASE WHEN c.retained_90d     THEN 1 ELSE 0 END                          AS retained_90d
    FROM customers c
    CROSS JOIN bounds b
    ORDER BY c.CustomerID;
""").df()

print("shape:", customer_summary.shape)
customer_summary.head()

shape: (5878, 14)


,customer_id,cohort_month,country,first_purchase_date,last_purchase_date,customer_lifespan_days,total_orders,total_revenue,avg_order_value,first_order_value,is_churned,is_lapsed_90d,eligible_for_90d,retained_90d
0,12346,2009-12-01,United Kingdom,2009-12-14,2011-01-18,400,12,77556.46,6463.04,45.00,0,1,1,1
1,12347,2010-10-01,Iceland,2010-10-31,2011-12-07,402,8,5633.32,704.16,611.53,0,0,1,1
2,12348,2010-09-01,Finland,2010-09-27,2011-09-25,363,5,2019.40,403.88,222.16,0,0,1,1
3,12349,2010-04-01,Italy,2010-04-29,2011-11-21,571,4,4428.69,1107.17,1068.52,0,0,1,1
4,12350,2011-02-01,Norway,2011-02-02,2011-02-02,0,1,334.40,334.40,334.40,1,1,1,0


In [5]:
# Quick sanity check — segment counts should match what Phase 4 reported
customer_summary[["is_churned", "is_lapsed_90d", "eligible_for_90d", "retained_90d"]].sum().to_frame("customers")

,customers
is_churned,1623
is_lapsed_90d,2989
eligible_for_90d,5281
retained_90d,2721


In [6]:
out = TABLEAU_DIR / "customer_summary.csv"
customer_summary.to_csv(out, index=False, date_format="%Y-%m-%d")
print("Saved →", out, f"({out.stat().st_size/1024:.1f} KB)")

Saved →

 C:\Users\Pitu\Desktop\Claude\Cohort & Retention Analysis\tableau\customer_summary.csv (509.1 KB)


## 3. Data dictionary

Reference for whoever opens the workbook later (including future you).

In [7]:
dictionary = pd.DataFrame([
    # cohort_matrix.csv
    ("cohort_matrix.csv", "cohort_month",      "DATE",  "Month each customer in this cohort first purchased"),
    ("cohort_matrix.csv", "cohort_index",      "INT",   "Months since cohort_month (0 = acquisition month)"),
    ("cohort_matrix.csv", "cohort_size",       "INT",   "Unique customers in the cohort (constant per cohort)"),
    ("cohort_matrix.csv", "active_customers",  "INT",   "Unique customers from the cohort active in this month"),
    ("cohort_matrix.csv", "retention_pct",     "FLOAT", "100 × active_customers / cohort_size"),
    # customer_summary.csv
    ("customer_summary.csv", "customer_id",             "INT",   "Unique customer identifier"),
    ("customer_summary.csv", "cohort_month",            "DATE",  "Month of first purchase — use to filter / colour by cohort"),
    ("customer_summary.csv", "country",                 "STR",   "Shipping country on first order"),
    ("customer_summary.csv", "first_purchase_date",     "DATE",  "Date of first ever order"),
    ("customer_summary.csv", "last_purchase_date",      "DATE",  "Date of most recent order"),
    ("customer_summary.csv", "customer_lifespan_days",  "INT",   "Days between first and last purchase (0 for one-and-done)"),
    ("customer_summary.csv", "total_orders",            "INT",   "Lifetime distinct invoice count"),
    ("customer_summary.csv", "total_revenue",           "FLOAT", "Lifetime £ spent"),
    ("customer_summary.csv", "avg_order_value",         "FLOAT", "total_revenue / total_orders"),
    ("customer_summary.csv", "first_order_value",       "FLOAT", "£ spent on first order (predictor of future retention)"),
    ("customer_summary.csv", "is_churned",              "BOOL",  "1 if total_orders = 1 (brief's definition); else 0"),
    ("customer_summary.csv", "is_lapsed_90d",           "BOOL",  "1 if no purchase in last 90 days of dataset"),
    ("customer_summary.csv", "eligible_for_90d",        "BOOL",  "1 if acquired ≥ 90 days before dataset end (controls for right-censoring)"),
    ("customer_summary.csv", "retained_90d",            "BOOL",  "1 if customer placed another order within 90 days of first (only meaningful when eligible_for_90d=1)"),
], columns=["file", "column", "type", "meaning"])

dictionary.to_csv(TABLEAU_DIR / "data_dictionary.csv", index=False)
dictionary

,file,column,type,meaning
0,cohort_matrix.csv,cohort_month,DATE,Month each customer in this cohort first purch...
1,cohort_matrix.csv,cohort_index,INT,Months since cohort_month (0 = acquisition month)
2,cohort_matrix.csv,cohort_size,INT,Unique customers in the cohort (constant per c...
3,cohort_matrix.csv,active_customers,INT,Unique customers from the cohort active in thi...
4,cohort_matrix.csv,retention_pct,FLOAT,100 × active_customers / cohort_size
5,customer_summary.csv,customer_id,INT,Unique customer identifier
6,customer_summary.csv,cohort_month,DATE,Month of first purchase — use to filter / colo...
7,customer_summary.csv,country,STR,Shipping country on first order
8,customer_summary.csv,first_purchase_date,DATE,Date of first ever order
9,customer_summary.csv,last_purchase_date,DATE,Date of most recent order


## 4. Final folder listing

Everything Tableau needs lives in `tableau/`.

In [8]:
for p in sorted(TABLEAU_DIR.glob("*.csv")):
    print(f"  {p.name:<30} {p.stat().st_size/1024:>8.1f} KB")

  cohort_matrix.csv                   8.7 KB
  customer_summary.csv              509.1 KB
  data_dictionary.csv                 1.6 KB


## Next — Build the dashboard in Tableau Desktop

1. **Connect** → *Text file* → pick `tableau/cohort_matrix.csv`.
2. **Add another connection** → *Text file* → `tableau/customer_summary.csv`.
3. Set `cohort_month` to a **Date** in both files (Tableau usually infers correctly thanks to ISO format).
4. Sheets to build:
   - **Cohort heatmap** — rows: `cohort_month` (discrete, month), cols: `cohort_index`, colour & label: `AVG(retention_pct)`.
   - **Country bar** — rows: `country`, bars: 90-day retention computed from `customer_summary` (use `eligible_for_90d=1` filter).
   - **Acquisition seasonality** — line chart of avg `retention_90d` by `MONTH(cohort_month)`.
   - **KPI cards** — big numbers for one-and-done %, average orders, total revenue.
5. Combine into a single dashboard with a title bar quoting the three findings from Phase 4.